In [ ]:
!nvidia-smi
![ -d "SE374.R11-Group11" ] && (cd SE374.R11-Group11 && git pull) || git clone https://github.com/EmmYeuuKhoaaHocc/SE374.R11-Group11.git
%cd /content/SE374.R11-Group11/apps/worker

In [ ]:
!pip install -q -r requirements.txt

In [ ]:
from getpass import getpass

# Enter your ngrok TCP endpoint exposed for local RabbitMQ port 5672 (e.g. 0.tcp.ap.ngrok.io and 12345)
rmq_host = input("RabbitMQ ngrok host (e.g. 0.tcp.ap.ngrok.io): ").strip()
rmq_port = input("RabbitMQ ngrok TCP port: ").strip()
rmq_user = input("RabbitMQ username: ").strip()
rmq_pass = getpass("RabbitMQ password: ")
print(f"Configured RabbitMQ target: {rmq_host}:{rmq_port} as {rmq_user}")

In [ ]:
import os
import subprocess
import time
import urllib.request

env = os.environ.copy()
env["RABBITMQ_USER"] = rmq_user
env["RABBITMQ_PASS"] = rmq_pass

subprocess.run(["fuser", "-k", "8000/tcp"], stderr=subprocess.DEVNULL)

log_file = open("/content/SE374.R11-Group11/apps/worker/worker.log", "w")
proc = subprocess.Popen(
    [
        "python", "main.py",
        "--host", "0.0.0.0",
        "--port", "8000",
        "--mc-batch-size", "5",
        "--rmq-host", rmq_host,
        "--rmq-port", str(rmq_port),
    ],
    cwd="/content/SE374.R11-Group11/apps/worker",
    env=env,
    stdout=log_file,
    stderr=subprocess.STDOUT,
)

time.sleep(6)
with open("/content/SE374.R11-Group11/apps/worker/worker.log") as f:
    print(f.read())

with urllib.request.urlopen("http://127.0.0.1:8000/health") as resp:
    print("Health check:", resp.read().decode())